In [2]:
import sys
sys.path.append("..")  #  the notebook runs inside notebook/, and src/ is one folder up

from src.normalize import norm_name

print(repr(norm_name("Récord Office (Pvt.) Ltd")))   # expect 'record office'
print(repr(norm_name("फर्स्ट फूड प्राइवेट लिमिटेड")))      # expect ''
print(repr(norm_name("FH  FH Business, LLC")))        # expect 'fh fh business'

'record office'
''
'fh fh business'


In [3]:
from src.normalize import norm_address

print(sorted(norm_address("88 OLIVE CIR, LEBANON, TN")))
print(norm_address("88 Olive Circle, Lebanon, TN") == norm_address("Lebanon, Tennessee, 88 Olive Cir"))
print(sorted(norm_address("CALUMET CITY, 351 HOXIE AVE, N/A, IL")))
print(repr(norm_address(float("nan"))))

['88', 'circle', 'lebanon', 'olive', 'tennessee']
True
['351', 'avenue', 'calumet', 'city', 'hoxie', 'illinois']
frozenset()


In [4]:
from src.blocking import name_keys

print(sorted(name_keys("Record Office Pvt Ltd", "US")))
print(sorted(name_keys("Rcdbr Offie", "US")))
print(sorted(name_keys("FH FH Business", "US")))
print(name_keys("राजस्थान स्टोर", "IN"))

['n|US|office', 'n|US|record', 'p|US|offi', 'p|US|reco']
['n|US|offie', 'n|US|rcdbr', 'p|US|offi', 'p|US|rcdb']
['n|US|business', 'p|US|busi']
set()


In [5]:
from src.blocking import address_keys

print(sorted(address_keys("CALUMET CITY, 351 HOXIE AVE, N/A, IL", "US")))
print(sorted(address_keys("88 OLIVE CIR, LEBANON, TN", "US")))
print(address_keys(float("nan"), "US"))
print(address_keys("Lebanon, Tennessee", "US"))

['a|US|351|avenue', 'a|US|351|calumet', 'a|US|351|city', 'a|US|351|hoxie', 'a|US|351|illinois']
['a|US|88|circle', 'a|US|88|lebanon', 'a|US|88|olive', 'a|US|88|tennessee']
set()
set()


In [6]:
import sys; sys.path.append("..")
import pandas as pd

%load_ext autoreload
# WHY: re-imports src/ after you edit blocking.py, so no kernel restart is needed
%autoreload 2

from src.blocking import build_index, block_sizes

# WHY: replace the file names below with the real ones from the dataset/dev listing
s2_dev = pd.read_csv("../dataset/dev/s2_dev.tsv", sep="\t")
s3_dev = pd.read_csv("../dataset/dev/s3_dev.tsv", sep="\t")

pool = pd.concat([s2_dev, s3_dev], ignore_index=True)
print(len(pool), pool.columns.tolist())   # expect 467494 rows

index = build_index(pool)
sizes = block_sizes(index)

print("number of keys:", len(sizes))
print(sizes.describe(percentiles=[0.5, 0.9, 0.99, 0.999]))
print(sizes.sort_values(ascending=False).head(15))

467494 ['entity_id', 'business_name', 'business_address', 'country']
number of keys: 1153212
count    1.153212e+06
mean     4.447934e+00
std      5.851868e+01
min      1.000000e+00
50%      1.000000e+00
90%      5.000000e+00
99%      3.600000e+01
99.9%    4.050000e+02
max      1.462400e+04
dtype: float64
p|US|cent           14624
n|US|center         13202
p|US|part           12537
n|US|partners       12249
n|US|com            11641
p|India|indi        11406
n|US|and            11031
n|India|india       10778
p|US|serv           10557
p|US|grou            9558
n|US|group           9548
p|India|serv         9437
n|US|care            7407
p|US|asso            7383
n|India|services     7299
dtype: int64


In [7]:
import sys; sys.path.append("..")
import pandas as pd
from src.blocking import build_index, get_candidates

s1_dev = pd.read_csv("../dataset/dev/s1_dev.tsv", sep="\t")
gt_dev = pd.read_csv("../dataset/dev/gt_dev.tsv", sep="\t")
# WHY header=None: this file has no header row
val_ids = set(pd.read_csv("../dataset/dev/val_ids.txt", header=None)[0])

# WHY only validation entities: blocking is tuned and scored on these, never on the 80k training ones
val_s1 = s1_dev[s1_dev["entity_id"].isin(val_ids)]

# WHY NaN check: a singleton has no matches, so its true set is empty
truth = {
    sid: (set(m.split(",")) if isinstance(m, str) else set())
    for sid, m in zip(gt_dev["source1_entity_id"], gt_dev["matched_entity_ids"])
}

# WHY rebuild only if needed: skip these two lines if `index` is still in the kernel
s2 = pd.read_csv("../dataset/dev/s2_dev.tsv", sep="\t");
s3 = pd.read_csv("../dataset/dev/s3_dev.tsv", sep="\t")
index = build_index(pd.concat([s2, s3], ignore_index=True))

for cap in [50, 200, 1000]:
    found = total = n_cands = 0
    for eid, name, addr, country in zip(
        val_s1["entity_id"], val_s1["business_name"],
        val_s1["business_address"], val_s1["country"],
    ):
        cands = get_candidates(name, addr, country, index, cap)
        true = truth[eid]
        found += len(cands & true)
        total += len(true)
        n_cands += len(cands)
    print(f"cap={cap}: recall={found/total:.4f}, candidates per S1={n_cands/len(val_s1):.1f}")

cap=50: recall=0.8509, candidates per S1=39.8
cap=200: recall=0.8951, candidates per S1=165.6
cap=1000: recall=0.9529, candidates per S1=809.9


In [8]:
#top k candidates based on the number of shared keys
from src.blocking import get_candidates_topk
for k in [20, 30, 50]:
    found = total = n_cands = 0
    for eid, name, addr, country in zip(
        val_s1["entity_id"], val_s1["business_name"],
        val_s1["business_address"], val_s1["country"],
    ):
        cands = get_candidates_topk(name, addr, country, index, 1000, k)
        true = truth[eid]
        found += len(cands & true)
        total += len(true)
        n_cands += len(cands)
    print(f"cap=1000, k={k}: recall={found/total:.4f}, candidates per S1={n_cands/len(val_s1):.1f}")
print("\n")

cap=1000, k=20: recall=0.8644, candidates per S1=19.4
cap=1000, k=30: recall=0.8769, candidates per S1=28.9
cap=1000, k=50: recall=0.8905, candidates per S1=47.5




In [9]:
#based on top k candidates but unique have more wieghtage
for cap in [200, 1000]:
    for k in [20, 30]:
        found = total = n_cands = 0
        for eid, name, addr, country in zip(
            val_s1["entity_id"], val_s1["business_name"],
            val_s1["business_address"], val_s1["country"],
        ):
            cands = get_candidates_topk(name, addr, country, index, cap, k)
            true = truth[eid]
            found += len(cands & true)
            total += len(true)
            n_cands += len(cands)
        print(f"cap={cap}, k={k}: recall={found/total:.4f}, candidates per S1={n_cands/len(val_s1):.1f}")

cap=200, k=20: recall=0.8459, candidates per S1=17.5
cap=200, k=30: recall=0.8574, candidates per S1=25.4
cap=1000, k=20: recall=0.8644, candidates per S1=19.4
cap=1000, k=30: recall=0.8769, candidates per S1=28.9


In [10]:
from src.blocking import get_candidates, get_candidates_topk

# WHY set_index: lets us look up a missed record's name and address by its ID
pool = pd.concat([s2, s3], ignore_index=True).set_index("entity_id")

sample = val_s1.sample(300, random_state=1)
shown = 0
for eid, name, addr, country in zip(
    sample["entity_id"], sample["business_name"],
    sample["business_address"], sample["country"],
):
    true = truth[eid]
    if not true:
        continue
    full = get_candidates(name, addr, country, index, 1000)
    top = get_candidates_topk(name, addr, country, index, 1000, 30)
    for t in true - top:
        print(f"S1   : {name} | {addr}")
        print(f"MISS : {pool.loc[t, 'business_name']} | {pool.loc[t, 'business_address']}")
        # WHY this flag: True = ranking lost it, False = blocking never found it
        print(f"in full cap=1000 set: {t in full}\n")
        shown += 1
        break          # one miss per entity, for variety
    if shown == 10:
        break

S1   : Real Laxmi Business Limited | Fortuna-B- Wing, Flat No. 604, Hiranandani Estate, Patlipada, Thane, Maharashtra
MISS : रियल लक्ष्मी बिजनेस लिमिटेड | Thane, MH, Fortuna-b- Wing
in full cap=1000 set: False

S1   : New Delhi Sciences Private Limited | 7149, D-7 Vasant Kunj, New Delhi, Delhi
MISS : delhisciences.com | 07149, D-7 Vasant Kunj, New Delhi, DL
in full cap=1000 set: True

S1   : Maa Tirupati Media Private Limited | Maharashtra, Chsl 100Ft Rd, Thane, Vasai, Flat1 B12 Saraswati Bldg
MISS : मां तिरुपति मीडिया प्राइवेट लिमिटेड | FLAT##1 B12 SARASWATI BLG, THANE, महाराष्ट्र
in full cap=1000 set: False

S1   : Shyam Marketing Private Limited | New Delhi, 9Th Floor, Pearls Omaxe Tower A, Netaji Subhash Place, Pitampura, 1009, North Delhi, Delhi
MISS : Shyam Marketing Private | nan
in full cap=1000 set: True

S1   : Smart Indo Investment Private Limited | New No.21, Old No.109, Rasappa Chetty Street Park Town, Chennai, Tamil Nadu
MISS : ஸ்மார்ட் இண்டோ இன்வெஸ்ட்மெண்ட் பிரைவேட் லிமி

In [11]:
from src.blocking import get_candidates_topk, prepare_pool, rerank

# WHY a new name: the earlier `pool` was changed by set_index
pool_df = pd.concat([s2, s3], ignore_index=True)
prepared = prepare_pool(pool_df)

found100 = found20 = found30 = total = n100 = 0
for eid, name, addr, country in zip(
    val_s1["entity_id"], val_s1["business_name"],
    val_s1["business_address"], val_s1["country"],
):
    true = truth[eid]
    first = get_candidates_topk(name, addr, country, index, 1000, 100)
    found100 += len(first & true)
    n100 += len(first)
    found20 += len(rerank(name, addr, first, prepared, 20) & true)
    found30 += len(rerank(name, addr, first, prepared, 30) & true)
    total += len(true)

print(f"first stage top-100: recall={found100/total:.4f}, candidates={n100/len(val_s1):.1f}")
print(f"reranked k=20: recall={found20/total:.4f}")
print(f"reranked k=30: recall={found30/total:.4f}")

first stage top-100: recall=0.9087, candidates=92.3
reranked k=20: recall=0.8928
reranked k=30: recall=0.8958
